In [1]:
import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageOps
from scipy.ndimage import median_filter as scipy_median
from skimage import exposure
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import roc_curve, auc, precision_recall_curve, classification_report

warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ==============================================================
# CONFIG  — set your paths here
# ==============================================================
SPLITS_CSV = "/kaggle/input/datasets/adaisdiashdh/splits-dataset/pet_stage2_splits.csv"   # ← set your PET splits CSV path
OUT_DIR    = "/kaggle/working"

# Fix stale path prefix baked into the CSV if needed.
# Set both to the same value if paths are already correct.
OLD_PATH_PREFIX = "/kaggle/input/datasets/adaisdiashdh/mri-pet-slices"
NEW_PATH_PREFIX = "/kaggle/input/datasets/adaisdiashdh/mri-pet-slices"

CFG = dict(
    epochs      = 10,       # keep short — search, not final training
    patience    = 4,        # early stopping on val ROC-AUC
    batch_size  = 32,
    lr          = 5e-5,
    dense_units = 128,
    dropout     = 0.5,
    img_size    = 224,
    num_workers = 2,
)

LABEL_MAP  = {"AD": 1, "CN": 0}
IDX2LABEL  = {0: "CN", 1: "AD"}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ==============================================================
# 1.  PREPROCESSING PRIMITIVES
#     All functions: PIL Image (RGB) → PIL Image (RGB)
#
#     Section A — carried over from MRI search (unchanged)
#     Section B — PET-specific ops
# ==============================================================

# ── Section A: MRI ops (carried over unchanged) ───────────────

def op_identity(img: Image.Image) -> Image.Image:
    """No preprocessing — raw slice."""
    return img


def op_clahe(img: Image.Image) -> Image.Image:
    """
    CLAHE on the L channel of LAB colour space.
    Clip limit 2.0, tile 8×8.
    """
    arr = np.array(img.convert("RGB"))
    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge([l, a, b])
    out = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    return Image.fromarray(out)


def op_histogram_eq(img: Image.Image) -> Image.Image:
    """Global histogram equalization on grayscale, replicated to RGB."""
    gray = np.array(img.convert("L"))
    eq   = exposure.equalize_hist(gray)
    eq8  = (eq * 255).astype(np.uint8)
    rgb  = np.stack([eq8, eq8, eq8], axis=-1)
    return Image.fromarray(rgb)


def op_zscore(img: Image.Image) -> Image.Image:
    """
    Per-slice z-score normalisation clipped to ±3σ then rescaled to [0,255].
    Removes scanner-level intensity bias.
    """
    arr  = np.array(img.convert("L")).astype(np.float32)
    mean, std = arr.mean(), arr.std()
    if std < 1e-6:
        std = 1.0
    norm = np.clip((arr - mean) / std, -3, 3)
    norm = ((norm + 3) / 6 * 255).astype(np.uint8)
    rgb  = np.stack([norm, norm, norm], axis=-1)
    return Image.fromarray(rgb)


def op_minmax(img: Image.Image) -> Image.Image:
    """Per-slice min-max normalisation to [0,255]."""
    arr = np.array(img.convert("L")).astype(np.float32)
    lo, hi = arr.min(), arr.max()
    if hi - lo < 1e-6:
        hi = lo + 1.0
    norm = ((arr - lo) / (hi - lo) * 255).astype(np.uint8)
    rgb  = np.stack([norm, norm, norm], axis=-1)
    return Image.fromarray(rgb)


def op_gaussian_blur(img: Image.Image) -> Image.Image:
    """Mild Gaussian blur (radius=1) to reduce noise."""
    return img.filter(ImageFilter.GaussianBlur(radius=1))


def op_median_filter(img: Image.Image) -> Image.Image:
    """3×3 median filter — removes salt-and-pepper noise."""
    arr  = np.array(img.convert("L"))
    filt = scipy_median(arr, size=3).astype(np.uint8)
    rgb  = np.stack([filt, filt, filt], axis=-1)
    return Image.fromarray(rgb)


def op_gamma_dark(img: Image.Image) -> Image.Image:
    """Gamma=0.75 — brightens dark slices."""
    arr  = np.array(img.convert("L")).astype(np.float32) / 255.0
    out  = (np.power(arr, 0.75) * 255).astype(np.uint8)
    rgb  = np.stack([out, out, out], axis=-1)
    return Image.fromarray(rgb)


def op_gamma_bright(img: Image.Image) -> Image.Image:
    """Gamma=1.5 — darkens bright background."""
    arr  = np.array(img.convert("L")).astype(np.float32) / 255.0
    out  = (np.power(arr, 1.5) * 255).astype(np.uint8)
    rgb  = np.stack([out, out, out], axis=-1)
    return Image.fromarray(rgb)


def op_edge_enhance(img: Image.Image) -> Image.Image:
    """Unsharp masking to enhance tissue boundaries."""
    arr      = np.array(img.convert("L")).astype(np.float32)
    blurred  = cv2.GaussianBlur(arr, (0, 0), sigmaX=2)
    sharp    = np.clip(arr + 0.5 * (arr - blurred), 0, 255).astype(np.uint8)
    rgb = np.stack([sharp, sharp, sharp], axis=-1)
    return Image.fromarray(rgb)


def op_bilateral(img: Image.Image) -> Image.Image:
    """Bilateral filter — smooths homogeneous regions, preserves edges."""
    arr  = np.array(img.convert("L"))
    filt = cv2.bilateralFilter(arr, d=9, sigmaColor=75, sigmaSpace=75)
    rgb  = np.stack([filt, filt, filt], axis=-1)
    return Image.fromarray(rgb)


# ── Section B: PET-specific ops ───────────────────────────────

def op_suv_clip(img: Image.Image) -> Image.Image:
    """
    SUV window clipping to [0, 4] SUV range.

    PET images are stored as 8-bit PNGs but represent SUV values.
    Pixel intensity 255 typically encodes the scanner max (~6–10 SUV).
    Clipping at 4/6 × 255 ≈ 170 suppresses hot-spot outliers (bladder,
    kidneys) that dominate the histogram and obscure cortical uptake —
    the region most relevant for AD diagnosis.

    Clip threshold: 4/6 of full scale ≈ 170 (assumes max SUV ≈ 6).
    After clipping, rescale to [0,255] so the dynamic range is preserved
    for the downstream normalisation ops.
    """
    arr  = np.array(img.convert("L")).astype(np.float32)
    clip_val = 170.0           # ≈ SUV 4 assuming max SUV ≈ 6 → 255
    arr  = np.clip(arr, 0, clip_val)
    arr  = (arr / clip_val * 255).astype(np.uint8)
    rgb  = np.stack([arr, arr, arr], axis=-1)
    return Image.fromarray(rgb)


def op_suv_clip_tight(img: Image.Image) -> Image.Image:
    """
    Tighter SUV window clip to [0, 2.5] SUV range.

    A tighter window (≈ 106/255) emphasises subtle cortical differences
    between AD (hypometabolism) and CN (normal uptake) at the cost of
    saturating high-uptake structures.  Useful as a complement to the
    standard clip to see which window the model prefers.
    """
    arr  = np.array(img.convert("L")).astype(np.float32)
    clip_val = 106.0           # ≈ SUV 2.5 assuming max SUV ≈ 6 → 255
    arr  = np.clip(arr, 0, clip_val)
    arr  = (arr / clip_val * 255).astype(np.uint8)
    rgb  = np.stack([arr, arr, arr], axis=-1)
    return Image.fromarray(rgb)


def op_hot_colormap(img: Image.Image) -> Image.Image:
    """
    Apply the 'hot' false-colour LUT (black→red→yellow→white).

    Nuclear medicine readers routinely view PET in the hot colourmap.
    Converting grayscale slices to hot gives VGG19 (trained on colour
    images) colour gradients that encode metabolic intensity, which may
    be more discriminative than uniform grayscale intensity alone.

    Implementation: use OpenCV COLORMAP_HOT on the L channel.
    Output is a genuine 3-channel RGB image (not stacked grayscale).
    """
    arr  = np.array(img.convert("L"))
    hot  = cv2.applyColorMap(arr, cv2.COLORMAP_HOT)          # BGR
    rgb  = cv2.cvtColor(hot, cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)


def op_jet_colormap(img: Image.Image) -> Image.Image:
    """
    Apply the 'jet' false-colour LUT (blue→green→red).

    Jet is the most common PET display colourmap in clinical software
    (e.g. SPM, MIM).  Encodes low uptake as blue and high as red,
    giving the CNN a strong colour cue for hypometabolic regions.
    """
    arr  = np.array(img.convert("L"))
    jet  = cv2.applyColorMap(arr, cv2.COLORMAP_JET)           # BGR
    rgb  = cv2.cvtColor(jet, cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)


def op_log_compress(img: Image.Image) -> Image.Image:
    """
    Logarithmic intensity compression: out = log(1 + pixel) / log(256) × 255

    PET intensity histograms are strongly right-skewed (most brain voxels
    cluster near background; a few hot spots dominate).  Log compression
    redistributes dynamic range toward the darker cortical uptake values
    where the AD signal lives, without discarding any information.
    """
    arr  = np.array(img.convert("L")).astype(np.float32)
    log_arr = np.log1p(arr) / np.log(256) * 255.0
    out  = np.clip(log_arr, 0, 255).astype(np.uint8)
    rgb  = np.stack([out, out, out], axis=-1)
    return Image.fromarray(rgb)


def op_background_mask(img: Image.Image) -> Image.Image:
    """
    Zero out background voxels below Otsu threshold.

    PET slices contain scanner-bed noise and low-level background counts
    that are uninformative.  Otsu's method finds the optimal threshold
    separating background from brain tissue, and all sub-threshold pixels
    are set to zero.  This is analogous to skull-stripping in MRI.
    """
    arr  = np.array(img.convert("L"))
    _, mask = cv2.threshold(arr, 0, 255,
                            cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    masked = cv2.bitwise_and(arr, arr, mask=mask)
    rgb    = np.stack([masked, masked, masked], axis=-1)
    return Image.fromarray(rgb)


def op_percentile_stretch(img: Image.Image) -> Image.Image:
    """
    Percentile contrast stretch: rescale [p2, p98] → [0, 255].

    More robust than global min-max because PET slices often contain a
    handful of extreme bright pixels (bladder, kidneys) that compress the
    rest of the histogram.  Stretching between the 2nd and 98th percentiles
    gives a stable, outlier-resistant normalisation.
    """
    arr  = np.array(img.convert("L")).astype(np.float32)
    p2, p98 = np.percentile(arr, 2), np.percentile(arr, 98)
    if p98 - p2 < 1e-6:
        p98 = p2 + 1.0
    stretched = np.clip((arr - p2) / (p98 - p2) * 255, 0, 255).astype(np.uint8)
    rgb  = np.stack([stretched, stretched, stretched], axis=-1)
    return Image.fromarray(rgb)


# ── Master op registry ────────────────────────────────────────
ALL_OPS = {
    # MRI ops (carried over)
    "identity":           op_identity,
    "clahe":              op_clahe,
    "histogram_eq":       op_histogram_eq,
    "zscore":             op_zscore,
    "minmax":             op_minmax,
    "gaussian_blur":      op_gaussian_blur,
    "median_filter":      op_median_filter,
    "gamma_dark":         op_gamma_dark,
    "gamma_bright":       op_gamma_bright,
    "edge_enhance":       op_edge_enhance,
    "bilateral":          op_bilateral,
    # PET-specific ops
    "suv_clip":           op_suv_clip,
    "suv_clip_tight":     op_suv_clip_tight,
    "hot_colormap":       op_hot_colormap,
    "jet_colormap":       op_jet_colormap,
    "log_compress":       op_log_compress,
    "background_mask":    op_background_mask,
    "percentile_stretch": op_percentile_stretch,
}

# ==============================================================
# 2.  RECIPES
#     Group A–H  : identical to MRI search (direct comparability)
#     Group P     : PET-specific recipes (new)
# ==============================================================

RECIPES = {
    # ── Group A: Baseline ──────────────────────────────────────
    "A01_raw":
        [],

    # ── Group B: Single normalisation ─────────────────────────
    "B01_clahe":
        ["clahe"],
    "B02_histogram_eq":
        ["histogram_eq"],
    "B03_zscore":
        ["zscore"],
    "B04_minmax":
        ["minmax"],
    "B05_gamma_dark":
        ["gamma_dark"],
    "B06_gamma_bright":
        ["gamma_bright"],

    # ── Group C: Single denoising ──────────────────────────────
    "C01_gaussian_blur":
        ["gaussian_blur"],
    "C02_median_filter":
        ["median_filter"],
    "C03_bilateral":
        ["bilateral"],

    # ── Group D: Single sharpening ─────────────────────────────
    "D01_edge_enhance":
        ["edge_enhance"],

    # ── Group E: Denoise → Normalise ──────────────────────────
    "E01_gaussian_clahe":
        ["gaussian_blur", "clahe"],
    "E02_median_clahe":
        ["median_filter", "clahe"],
    "E03_bilateral_clahe":
        ["bilateral", "clahe"],
    "E04_gaussian_zscore":
        ["gaussian_blur", "zscore"],
    "E05_median_zscore":
        ["median_filter", "zscore"],
    "E06_bilateral_zscore":
        ["bilateral", "zscore"],
    "E07_bilateral_minmax":
        ["bilateral", "minmax"],
    "E08_gaussian_minmax":
        ["gaussian_blur", "minmax"],

    # ── Group F: Normalise → Sharpen ──────────────────────────
    "F01_clahe_edge":
        ["clahe", "edge_enhance"],
    "F02_zscore_edge":
        ["zscore", "edge_enhance"],
    "F03_bilateral_clahe_edge":
        ["bilateral", "clahe", "edge_enhance"],

    # ── Group G: Gamma → Normalise ────────────────────────────
    "G01_gamma_dark_clahe":
        ["gamma_dark", "clahe"],
    "G02_gamma_dark_zscore":
        ["gamma_dark", "zscore"],
    "G03_gamma_bright_clahe":
        ["gamma_bright", "clahe"],

    # ── Group H: Full 3-step MRI pipelines ────────────────────
    "H01_bilateral_clahe_edge":
        ["bilateral", "clahe", "edge_enhance"],
    "H02_bilateral_zscore_edge":
        ["bilateral", "zscore", "edge_enhance"],
    "H03_gamma_dark_bilateral_clahe":
        ["gamma_dark", "bilateral", "clahe"],
    "H04_median_clahe_edge":
        ["median_filter", "clahe", "edge_enhance"],
    "H05_gamma_dark_bilateral_zscore":
        ["gamma_dark", "bilateral", "zscore"],

    # ── Group P: PET-specific recipes ─────────────────────────
    # P1x — SUV windowing (the single most PET-specific op)
    "P01_suv_clip":
        ["suv_clip"],
    "P02_suv_clip_tight":
        ["suv_clip_tight"],

    # P2x — False-colour colourmap (PET is routinely viewed in colour)
    "P03_hot_colormap":
        ["hot_colormap"],
    "P04_jet_colormap":
        ["jet_colormap"],

    # P3x — Distribution correction ops
    "P05_log_compress":
        ["log_compress"],
    "P06_percentile_stretch":
        ["percentile_stretch"],
    "P07_background_mask":
        ["background_mask"],

    # P4x — SUV clip → normalise combos
    # Clip first to remove outlier hotspots, then enhance local contrast
    "P08_suv_clip_clahe":
        ["suv_clip", "clahe"],
    "P09_suv_clip_zscore":
        ["suv_clip", "zscore"],
    "P10_suv_clip_bilateral_clahe":
        ["suv_clip", "bilateral", "clahe"],

    # P5x — Log compress → normalise combos
    # Log first compresses the skewed PET distribution, then CLAHE enhances
    "P11_log_clahe":
        ["log_compress", "clahe"],
    "P12_log_bilateral_clahe":
        ["log_compress", "bilateral", "clahe"],

    # P6x — Background mask → normalise
    # Mask non-brain first, then normalise within the brain only
    "P13_mask_clahe":
        ["background_mask", "clahe"],
    "P14_mask_bilateral_clahe":
        ["background_mask", "bilateral", "clahe"],

    # P7x — Colourmap + normalise combos
    # After colourmap, CLAHE acts on the L channel of LAB — enhances
    # contrast while preserving the colour encoding
    "P15_hot_clahe":
        ["hot_colormap", "clahe"],
    "P16_jet_clahe":
        ["jet_colormap", "clahe"],

    # P8x — Full PET-tailored 3-step pipelines
    # Best MRI winner (H01) but with SUV clip prepended
    "P17_suv_bilateral_clahe_edge":
        ["suv_clip", "bilateral", "clahe", "edge_enhance"],
    # Percentile stretch + denoise + sharpen
    "P18_pct_bilateral_clahe":
        ["percentile_stretch", "bilateral", "clahe"],
    # Log + mask + clahe: normalise distribution, remove background, enhance
    "P19_log_mask_clahe":
        ["log_compress", "background_mask", "clahe"],
}

print(f"Total recipes: {len(RECIPES)}")
for name, ops in RECIPES.items():
    tag = " [PET]" if name.startswith("P") else ""
    print(f"  {name:<40} {ops}{tag}")

# ==============================================================
# 3.  DATASET
# ==============================================================

def apply_recipe(img: Image.Image, recipe: list) -> Image.Image:
    """Apply an ordered list of op names to a PIL image."""
    for op_name in recipe:
        img = ALL_OPS[op_name](img)
    return img


class PreprocessDataset(Dataset):
    def __init__(self, df, recipe: list, augment: bool = False, img_size: int = 224):
        self.df       = df.reset_index(drop=True)
        self.recipe   = recipe
        self.img_size = img_size
        if augment:
            self.post = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(brightness=0.1, contrast=0.1),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            ])
        else:
            self.post = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img    = Image.open(row["slice_path"]).convert("RGB")
        img    = apply_recipe(img, self.recipe)
        img    = img.convert("RGB")   # ensure RGB after all ops
        label  = LABEL_MAP[row["group"]]
        tensor = self.post(img)
        return tensor, label, row["subject_id"]


def make_loaders(df, recipe, cfg):
    loaders = {}
    for split in ["train", "val", "test"]:
        sub = df[df["split"] == split]
        ds  = PreprocessDataset(
            sub, recipe,
            augment=(split == "train"),
            img_size=cfg["img_size"],
        )
        loaders[split] = DataLoader(
            ds,
            batch_size=cfg["batch_size"],
            shuffle=(split == "train"),
            num_workers=cfg["num_workers"],
            pin_memory=True,
        )
    return loaders

# ==============================================================
# 4.  MODEL  (same VGG19Extractor as baseline — fair comparison)
# ==============================================================

class VGG19Extractor(nn.Module):
    def __init__(self, dense_units=128, dropout=0.5, num_classes=2):
        super().__init__()
        base          = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.gap      = nn.AdaptiveAvgPool2d(1)
        self.dropout  = nn.Dropout(dropout)
        self.fc1      = nn.Linear(512, dense_units)
        self.fc2      = nn.Linear(dense_units, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).view(x.size(0), -1)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# ==============================================================
# 5.  TRAINING LOOP
# ==============================================================

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs, all_subjects = [], [], [], []
    for imgs, labels, subjects in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        probs  = F.softmax(logits, dim=1)[:, 1]
        loss   = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_subjects.extend(subjects)
    return (total_loss / total, correct / total,
            np.array(all_preds), np.array(all_labels),
            np.array(all_probs), all_subjects)


def subject_agg(preds, labels, probs, subjects):
    """Mean-probability aggregation across slices → subject-level decision."""
    df = pd.DataFrame({"subject": subjects, "label": labels,
                        "pred": preds, "prob": probs})
    rows = []
    for subj, g in df.groupby("subject"):
        mean_prob  = g["prob"].mean()
        true_label = g["label"].iloc[0]
        rows.append({
            "subject":    subj,
            "true_label": true_label,
            "pred_label": int(mean_prob >= 0.5),
            "mean_prob":  mean_prob,
        })
    res     = pd.DataFrame(rows)
    acc     = (res["true_label"] == res["pred_label"]).mean()
    fpr, tpr, _ = roc_curve(res["true_label"], res["mean_prob"])
    su_roc  = auc(fpr, tpr)
    prec, rec, _ = precision_recall_curve(res["true_label"], res["mean_prob"])
    su_pr   = auc(rec, prec)
    return acc, su_roc, su_pr


def run_trial(recipe_name, recipe_ops, df, cfg, tmp_path):
    """
    Train one recipe for cfg['epochs'] epochs with early stopping on val ROC-AUC.
    Returns a dict of val + test metrics.
    """
    loaders   = make_loaders(df, recipe_ops, cfg)
    model     = VGG19Extractor(cfg["dense_units"], cfg["dropout"]).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg["lr"])

    best_val_roc = 0.0
    no_improve   = 0

    for epoch in range(1, cfg["epochs"] + 1):
        tr_loss, tr_acc = train_epoch(model, loaders["train"], optimizer, criterion)
        va_loss, va_acc, va_preds, va_lbls, va_probs, va_subjs = \
            eval_epoch(model, loaders["val"], criterion)

        fpr, tpr, _ = roc_curve(va_lbls, va_probs)
        va_roc = auc(fpr, tpr)

        if va_roc > best_val_roc:
            best_val_roc = va_roc
            no_improve   = 0
            torch.save(model.state_dict(), tmp_path)
        else:
            no_improve += 1
            if no_improve >= cfg["patience"]:
                print(f"    early stop @ ep {epoch}")
                break

        print(f"    ep {epoch:02d} | tr_acc {tr_acc:.3f} | "
              f"va_acc {va_acc:.3f} | va_roc {va_roc:.3f}"
              + (" ✅" if no_improve == 0 else ""))

    # ── Evaluate best checkpoint on val ──────────────────────
    model.load_state_dict(torch.load(tmp_path, map_location=DEVICE))
    _, sl_acc, sl_preds, sl_lbls, sl_probs, sl_subjs = \
        eval_epoch(model, loaders["val"], criterion)

    fpr, tpr, _  = roc_curve(sl_lbls, sl_probs)
    sl_roc       = auc(fpr, tpr)
    prec, rec, _ = precision_recall_curve(sl_lbls, sl_probs)
    sl_pr        = auc(rec, prec)
    su_acc, su_roc, su_pr = subject_agg(sl_preds, sl_lbls, sl_probs, sl_subjs)

    # ── Evaluate on test (reported, NOT used for ranking) ────
    _, tsl_acc, tsl_preds, tsl_lbls, tsl_probs, tsl_subjs = \
        eval_epoch(model, loaders["test"], criterion)
    fpr, tpr, _  = roc_curve(tsl_lbls, tsl_probs)
    tsl_roc      = auc(fpr, tpr)
    prec, rec, _ = precision_recall_curve(tsl_lbls, tsl_probs)
    tsl_pr       = auc(rec, prec)
    tsu_acc, tsu_roc, tsu_pr = subject_agg(tsl_preds, tsl_lbls, tsl_probs, tsl_subjs)

    return {
        "recipe":       recipe_name,
        "ops":          " → ".join(recipe_ops) if recipe_ops else "(none)",
        "pet_specific": recipe_name.startswith("P"),
        # Validation metrics (used for ranking)
        "val_sl_acc":   round(sl_acc,  4),
        "val_su_acc":   round(su_acc,  4),
        "val_sl_roc":   round(sl_roc,  4),
        "val_su_roc":   round(su_roc,  4),
        "val_sl_pr":    round(sl_pr,   4),
        "val_su_pr":    round(su_pr,   4),
        # Test metrics (reported only)
        "test_sl_acc":  round(tsl_acc,  4),
        "test_su_acc":  round(tsu_acc,  4),
        "test_sl_roc":  round(tsl_roc,  4),
        "test_su_roc":  round(tsu_roc,  4),
        "test_sl_pr":   round(tsl_pr,   4),
        "test_su_pr":   round(tsu_pr,   4),
    }

# ==============================================================
# 6.  RESULTS & PLOTS
# ==============================================================

def plot_results(results_df, out_dir):
    """
    Bar chart of val subject ROC-AUC.
    PET-specific recipes are highlighted in orange.
    MRI-carried-over recipes are in blue.
    Best recipe bar in red.
    """
    df = results_df.sort_values("val_su_roc", ascending=True).copy()
    best_val = df["val_su_roc"].max()

    colors = []
    for _, row in df.iterrows():
        if row["val_su_roc"] == best_val:
            colors.append("#e74c3c")      # red — best
        elif row.get("pet_specific", False):
            colors.append("#e67e22")      # orange — PET-specific
        else:
            colors.append("#3498db")      # blue — MRI-carried-over

    fig, ax = plt.subplots(figsize=(16, max(8, len(df) * 0.38)))
    bars = ax.barh(df["recipe"], df["val_su_roc"], color=colors, edgecolor="white")
    ax.set_xlabel("Val Subject ROC-AUC  (higher = better)")
    ax.set_title("PET Preprocessing Recipe Search — Val Subject ROC-AUC\n"
                 "🔴 Best  |  🟠 PET-specific  |  🔵 MRI-carried-over", fontsize=12)
    ax.axvline(best_val, color="#e74c3c", linestyle="--",
               linewidth=1, label=f"Best: {best_val:.3f}")

    for bar, val in zip(bars, df["val_su_roc"]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}", va="center", ha="left", fontsize=7.5)

    ax.legend(loc="lower right")
    ax.set_xlim(0, 1.10)
    plt.tight_layout()
    path = os.path.join(out_dir, "pet_preproc_results.png")
    plt.savefig(path, dpi=150)
    plt.show()
    print(f"  Plot saved: {path}")


def print_leaderboard(results_df):
    df = results_df.sort_values("val_su_roc", ascending=False).reset_index(drop=True)
    print("\n" + "=" * 100)
    print("PET PREPROCESSING SEARCH — LEADERBOARD  (ranked by val subject ROC-AUC)")
    print("=" * 100)
    header = (f"{'Rank':<5} {'Recipe':<40} {'Ops':<38} "
              f"{'Val SuROC':>9} {'Val SuAcc':>9} {'Test SuROC':>10} {'PET?':>5}")
    print(header)
    print("-" * 100)
    for i, row in df.iterrows():
        marker   = " ◀ BEST" if i == 0 else ""
        ops_str  = row["ops"][:37] if len(row["ops"]) > 37 else row["ops"]
        pet_flag = "✓" if row.get("pet_specific", False) else ""
        print(f"{i+1:<5} {row['recipe']:<40} {ops_str:<38} "
              f"{row['val_su_roc']:>9.4f} {row['val_su_acc']:>9.4f} "
              f"{row['test_su_roc']:>10.4f} {pet_flag:>5}{marker}")
    print("=" * 100)

    # Print PET-specific sub-leaderboard for quick inspection
    pet_df = df[df["pet_specific"] == True].reset_index(drop=True)
    if not pet_df.empty:
        print(f"\n  PET-specific recipes only ({len(pet_df)} recipes):")
        print(f"  {'Rank':<5} {'Recipe':<40} {'Val SuROC':>9} {'Test SuROC':>10}")
        print(f"  {'-'*65}")
        for i, row in pet_df.iterrows():
            marker = " ◀ BEST PET" if i == 0 else ""
            print(f"  {i+1:<5} {row['recipe']:<40} "
                  f"{row['val_su_roc']:>9.4f} {row['test_su_roc']:>10.4f}{marker}")

    return df

# ==============================================================
# 7.  MAIN
# ==============================================================

def main():
    # ── Load split CSV ────────────────────────────────────────
    print(f"\nLoading splits from: {SPLITS_CSV}")
    df = pd.read_csv(SPLITS_CSV)

    # ── Fix stale path prefix ─────────────────────────────────
    if OLD_PATH_PREFIX != NEW_PATH_PREFIX:
        before = df["slice_path"].iloc[0]
        df["slice_path"] = df["slice_path"].str.replace(
            OLD_PATH_PREFIX, NEW_PATH_PREFIX, regex=False)
        after = df["slice_path"].iloc[0]
        print(f"  Path prefix fix:")
        print(f"    before: {before}")
        print(f"    after : {after}")
    else:
        print("  Paths unchanged (OLD == NEW prefix)")

    print(f"  {len(df)} slice rows | "
          f"{df['subject_id'].nunique()} subjects | "
          f"splits: {df['split'].value_counts().to_dict()}")

    # ── Sanity check paths ────────────────────────────────────
    missing = [p for p in df["slice_path"].head(20) if not os.path.exists(p)]
    if missing:
        print(f"  ❌ WARNING: {len(missing)}/20 sample paths missing.")
        print(f"     Example: {missing[0]}")
        print(f"     Fix OLD_PATH_PREFIX / NEW_PATH_PREFIX in CONFIG.")
    else:
        print(f"  ✅ Path check passed (20/20 sample paths exist)")

    tmp_path    = os.path.join(OUT_DIR, "_pet_preproc_tmp.pth")
    all_results = []

    mri_count = sum(1 for n in RECIPES if not n.startswith("P"))
    pet_count = sum(1 for n in RECIPES if n.startswith("P"))
    print(f"\nRunning {len(RECIPES)} recipes "
          f"({mri_count} MRI-carried-over + {pet_count} PET-specific) "
          f"— {CFG['epochs']} epochs each\n")
    print("=" * 60)

    for idx, (recipe_name, recipe_ops) in enumerate(RECIPES.items()):
        ops_str  = " → ".join(recipe_ops) if recipe_ops else "(none)"
        pet_flag = " [PET-specific]" if recipe_name.startswith("P") else ""
        print(f"\n[{idx+1:02d}/{len(RECIPES)}] {recipe_name}{pet_flag}")
        print(f"  ops: {ops_str}")

        try:
            result = run_trial(recipe_name, recipe_ops, df, CFG, tmp_path)
            all_results.append(result)
            print(f"  val_su_roc={result['val_su_roc']:.4f} | "
                  f"val_su_acc={result['val_su_acc']:.4f} | "
                  f"test_su_roc={result['test_su_roc']:.4f}")
        except Exception as e:
            print(f"  ❌ FAILED: {e}")
            all_results.append({"recipe": recipe_name, "ops": ops_str,
                                  "pet_specific": recipe_name.startswith("P"),
                                  "val_su_roc": 0.0, "error": str(e)})

    # ── Results ───────────────────────────────────────────────
    results_df = pd.DataFrame(all_results).fillna(0)
    ranked_df  = print_leaderboard(results_df)

    csv_path = os.path.join(OUT_DIR, "pet_preproc_results.csv")
    ranked_df.to_csv(csv_path, index=False)
    print(f"\n  Full results saved: {csv_path}")

    plot_results(ranked_df, OUT_DIR)

    # ── Save best config as JSON ──────────────────────────────
    best = ranked_df.iloc[0]
    best_recipe_ops = RECIPES[best["recipe"]]
    best_config = {
        "best_recipe_name":  best["recipe"],
        "best_recipe_ops":   best_recipe_ops,
        "pet_specific":      bool(best.get("pet_specific", False)),
        "val_su_roc":        best["val_su_roc"],
        "val_su_acc":        best["val_su_acc"],
        "test_su_roc":       best["test_su_roc"],
        "test_su_acc":       best["test_su_acc"],
        # Best PET-specific recipe (may differ from overall best)
        "best_pet_specific_recipe": None,
        "note": (
            "To use in baseline: pass ops list to apply_recipe() "
            "inside your Dataset __getitem__ before torchvision transforms."
        ),
    }

    # Also record best PET-specific recipe separately
    pet_only = ranked_df[ranked_df["pet_specific"] == True]
    if not pet_only.empty:
        bp = pet_only.iloc[0]
        best_config["best_pet_specific_recipe"] = {
            "name":       bp["recipe"],
            "ops":        RECIPES[bp["recipe"]],
            "val_su_roc": bp["val_su_roc"],
            "test_su_roc": bp["test_su_roc"],
        }

    json_path = os.path.join(OUT_DIR, "pet_preproc_best_config.json")
    with open(json_path, "w") as f:
        json.dump(best_config, f, indent=2)
    print(f"\n  Best config saved: {json_path}")
    print(f"  Best recipe       : {best['recipe']}")
    print(f"  Best ops          : {best_recipe_ops}")
    print(f"  Val SuROC         : {best['val_su_roc']:.4f}")
    print(f"  Test SuROC        : {best['test_su_roc']:.4f}")
    if best_config["best_pet_specific_recipe"]:
        bpr = best_config["best_pet_specific_recipe"]
        print(f"\n  Best PET-specific : {bpr['name']}")
        print(f"  PET ops           : {bpr['ops']}")
        print(f"  PET Val SuROC     : {bpr['val_su_roc']:.4f}")

    # ── Usage instructions ────────────────────────────────────
    print("\n" + "=" * 60)
    print("HOW TO USE THE BEST RECIPE IN YOUR BASELINE")
    print("=" * 60)
    print("""
  1. Copy ALL_OPS dict and apply_recipe() into your baseline.

  2. In your Dataset __getitem__, add before the torchvision transforms:

       img = apply_recipe(img, best_recipe_ops)

  3. For PET colourmap ops (hot_colormap / jet_colormap):
     The output is already genuine RGB — no changes needed elsewhere.
     VGG19 will receive a 3-channel colour image as intended.

  4. That's it — rest of pipeline is unchanged.
""")

    if os.path.exists(tmp_path):
        os.remove(tmp_path)

    return ranked_df


if __name__ == "__main__":
    results = main()

Device : cuda
GPU    : Tesla T4
Total recipes: 49
  A01_raw                                  []
  B01_clahe                                ['clahe']
  B02_histogram_eq                         ['histogram_eq']
  B03_zscore                               ['zscore']
  B04_minmax                               ['minmax']
  B05_gamma_dark                           ['gamma_dark']
  B06_gamma_bright                         ['gamma_bright']
  C01_gaussian_blur                        ['gaussian_blur']
  C02_median_filter                        ['median_filter']
  C03_bilateral                            ['bilateral']
  D01_edge_enhance                         ['edge_enhance']
  E01_gaussian_clahe                       ['gaussian_blur', 'clahe']
  E02_median_clahe                         ['median_filter', 'clahe']
  E03_bilateral_clahe                      ['bilateral', 'clahe']
  E04_gaussian_zscore                      ['gaussian_blur', 'zscore']
  E05_median_zscore                        ['medi

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


  0%|          | 0.00/548M [00:00<?, ?B/s]

  1%|▏         | 8.00M/548M [00:00<00:06, 83.8MB/s]

  6%|▌         | 32.2M/548M [00:00<00:02, 184MB/s] 

...

    ep 06 | tr_acc 0.968 | va_acc 0.873 | va_roc 0.940


    ep 07 | tr_acc 0.970 | va_acc 0.852 | va_roc 0.930


    early stop @ ep 8


  val_su_roc=0.9890 | val_su_acc=0.9630 | test_su_roc=0.9807

[35/49] P05_log_compress [PET-specific]
  ops: log_compress


    ep 01 | tr_acc 0.552 | va_acc 0.628 | va_roc 0.640 ✅
